# 00 イントロ: Stable Diffusion 1.5 で画像を生成してみる

このノートブックでは **Stable Diffusion 1.5 (SD1.5)** を動かしながら、画像生成モデルの基盤である **「潜在拡散モデル(Latent Diffusion Model)」** の基本を体験します。

## 0. はじめに

### このノートブックでやること

1. [環境確認](#env) — デバイス (CPU / MPS / CUDA) とパッケージを確認する
2. [モデル読み込み](#load) — Hugging Face から `StableDiffusionPipeline` をダウンロードして読み込む
3. [画像生成](#gen) — prompt から 1 枚の画像を生成する
4. [パイプラインを覗いてみる](#peek) — components 一覧、パラメータ数、disk バイト数を概観
5. [パイプラインの中身を見る](#inside) — text encoder / UNet / VAE / scheduler を個別に観察
6. [tokenizer の動作を確認](#tokenizer) — prompt がどのように token 列になるか見てみる
7. [seed と steps を変えてみる](#sweep) — パラメータが生成に与える影響を観察する
8. [他のモデルを試す](#alts) — SD1.5 をベースに fine-tune した派生モデルを `pipe_alt` で読み込んで比較
9. [まとめ](#summary)

### 生成画像の保存

このノートブックで生成した画像は、notebook 内表示に加えて `outputs/00_sd15_intro/` ディレクトリに PNG として保存されます (§3 で 2 枚 [base + CPU 比較]、§7 で 3 枚 [seed / steps × 2]、§8 で 1 枚、計 6 枚。CPU 比較を skip した場合は 5 枚)。後から外部ビューア / image diff ツールで比較したい場合に便利です。

### Stable Diffusion 1.5 とは

- 2022 年に公開された **latent diffusion model** (Stable Diffusion 系の代表モデル)
- 画像そのものではなく、VAE で圧縮した **潜在空間 (latent space)** で diffusion を行う
- 主な構成要素は 4 つ:
  - **CLIP text encoder** — prompt を embedding に変換
  - **UNet** — latent から「予測ノイズ」を出力する denoiser
  - **VAE** — pixel ↔ latent の変換 (空間 1 辺を 1/8 に圧縮、channel は 3 → 4。値の総数では約 1/48 のサイズに圧縮される)
  - **scheduler** — diffusion step の刻み方を決める
- パラメータ規模は約 1.07 B (text_encoder 約 123 M + UNet 約 860 M + VAE 約 84 M、§4 で実測値を表示する。各コンポーネントの構造は §5 で個別に観察)

### 情報源 (現用 → 歴史的)

- HF model card (本ノートブックで使う 重み): <https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5> — SD1.5 の重み・config・tokenizer がここに集約されている。`from_pretrained(model_id)` の `model_id` はここのパスを指す
- PyPI (本ノートブックで使う library の配布元): <https://pypi.org/project/diffusers/> — PyPI (パイピーアイ) は `pip install diffusers` で取得する Python パッケージの配布元。version 一覧と release 履歴を確認できる (本ノートブックでは §1 で表示される `Diffusers : ...` のバージョンが入っている)
- GitHub (本ノートブックで使う library のソースコード): <https://github.com/huggingface/diffusers> — Hugging Face Diffusers。`from diffusers import StableDiffusionPipeline` の実装本体、pipeline の中身を読みたいときの参照先
- GitHub (reference 実装): <https://github.com/CompVis/stable-diffusion> — Stable Diffusion の original code (CompVis)。SD1.4 までの公式 release 元で、SD1.5 もこのコードベース上の fine-tune として release された
- GitHub (LDM 原典): <https://github.com/CompVis/latent-diffusion> — "High-Resolution Image Synthesis with Latent Diffusion Models" の論文 code、SD の数学的基盤

> SD1.5 を独自に release した RunwayML の GitHub repo (`runwayml/stable-diffusion`) は現在削除されています。SD1.5 の重み入手は HF Hub から、ソースコード参照は CompVis repo から、という運用が標準です。

### Hugging Face とは

このノートブックでモデルを取得する **Hugging Face** (以下 HF) は、機械学習モデル・データセット・デモアプリを共有・配布するプラットフォームです。GitHub の機械学習版に近いポジションで、研究者・企業がモデル重みと使い方をここで公開しています。

- モデル Hub: 各モデルが固有の `model_id` を持つ。SD1.5 の場合は [`stable-diffusion-v1-5/stable-diffusion-v1-5`](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5) (`<組織名>/<モデル名>` の形)。
- model card: 上記 URL のようなモデル説明ページ。学習データ・性能・ライセンス・サンプルコードが掲載されており、使う前に必ず目を通す対象。
- ライブラリ統合: `diffusers` / `transformers` の `from_pretrained(model_id)` を呼ぶだけで、重み・設定・tokenizer を HF Hub から自動取得し、`~/.cache/huggingface/hub/` にキャッシュする。2 回目以降は cache から読む。
  - Diffusers ライブラリは HF 自身が開発・配布 (GitHub link は上の「情報源」参照)。`StableDiffusionPipeline` をはじめ各種 model 用 pipeline を提供する。
- gated model: 一部のモデル (FLUX, SD3.5 など) は HF 上で利用条件への承認 + `hf auth login` での token 認証が必要。SD1.5 は承認不要で誰でも download できる。
- GitHub との役割分担: ソースコードは GitHub、モデル重み・使用例は HF Hub、というのが現代の ML エコシステムの定番構成 (モデル重みは数 GB 〜数十 GB あり Git で扱いにくいため、専用ホスティング側に分離した)。


## 0. 環境セットアップ（3環境 自動切替: Colabだけ pip / path も自動）


In [ ]:
# 3環境(Mac/Win/Colab)を同一ファイルで動かすための判定。Colabのみ pip（Mac/Winはenvに在るのでskip）。
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "transformers==5.9.0" "accelerate==1.13.0"
    print("Colab: pip done")
else:
    print("local(Mac/Win): pip skip（env利用）")

<a id="env"></a>

---
## 1. 環境確認

In [ ]:
import sys
import logging
import torch
import diffusers
import transformers

# HuggingFace Hub の認証警告を抑制する（cache から読む場合は不要なため）
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Diffusers   :", diffusers.__version__)
print("Transformers:", transformers.__version__)

# device と dtype を選ぶ
# - cuda → fp16 (NVIDIA GPU)
# - mps  → fp32 (Apple Silicon。fp16 だと SD1.5 の safety_checker が誤発火しやすい)
# - cpu  → fp32
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float32
else:
    device = "cpu"
    dtype = torch.float32

print("device      :", device)
print("dtype       :", dtype)

<a id="load"></a>

---
## 2. モデル読み込み

`StableDiffusionPipeline` を Hugging Face cache から読み込みます。今回 load する HF Hub レポジトリは [`stable-diffusion-v1-5/stable-diffusion-v1-5`](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5) です。

**初回実行は時間がかかります**:
- text encoder + UNet + VAE + safety_checker + tokenizer 設定 で **合計 約 5 GB** を Hugging Face Hub から自動 download (fp32 のデフォルト variant の場合。`variant="fp16"` を指定すれば半分程度の ~2.5 GB に)
- 回線速度次第で **数分〜30 分** 程度
- 保存先は `~/.cache/huggingface/hub/` (Hugging Face のデフォルト cache)
- **2 回目以降は cache から読むので数秒** に短縮

このノートブックでは **fp32 + safety_checker ON** という最も安全な設定で動かします（fp16 で動かすと MPS で safety_checker (CLIP) が誤発火して黒画像になることがあるため、入門時は fp32 が無難）。

### model_id

Diffusers は `model_id` を指定するだけで Hugging Face Hub から model を取得できます。
ここで設定しておきます。

In [ ]:
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
print("model_id:", MODEL_ID)

### Pipeline を読み込む

In [ ]:
import time

from diffusers import StableDiffusionPipeline

print("pipeline 読み込み中... (初回は約 5 GB の download を含むため数分〜30 分かかります)")
t_load = time.perf_counter()
pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype)
pipe = pipe.to(device)
load_elapsed = time.perf_counter() - t_load

# MPS ではメモリ圧で遅くなりやすいので attention slicing を有効化する
if device == "mps":
    pipe.enable_attention_slicing()
    print("[mps] enable_attention_slicing() 有効化")

print(f"load done in {load_elapsed:.2f} s  (cache hit なら数秒、初回 download 時は数分〜)")
print("pipeline クラス:", type(pipe).__name__)

<a id="gen"></a>

---
## 3. 画像生成

prompt を渡して 1 枚画像を生成します。

### 生成パラメータ、prompt、 seed を決める

In [ ]:
# 生成パラメータ (SD1.5 のネイティブ解像度)
width = 512
height = 512
num_inference_steps = 40       # UNet を呼ぶ回数
guidance_scale = 7.5           # classifier-free guidance の強さ

In [ ]:
# 古典的な SD1.5 デモプロンプト (参考):
# prompt = "a photo of an astronaut riding a horse"
# negative_prompt = ""

# 魔法使い (自然言語版)
prompt = (
    "a young witch with long flowing hair holding a glowing magic staff, "
    "in an enchanted forest with bioluminescent mushrooms, "
    "fantasy art, masterpiece, detailed"
)
negative_prompt = (
    "blurry, low quality, distorted, bad anatomy, extra limbs, "
    "watermark, signature"
)

# 魔法使い (tag 列挙版)
# prompt = (
#     "witch with magic staff, long hair, enchanted forest, glowing mushrooms, "
#     "bloom, high quality, very high resolution, colorful refraction, lens flare"
# )
# negative_prompt = (
#     "bad anatomy, bad hands, text, error, missing fingers, extra digit, "
#     "extra tail, fewer digits, multiple legs, malformation, close up"
# )

# 以前あったもの
# prompt = "high quality, very high resolution, large filesize, high detailed, distant wide shot, straight hair, dark hair, very long hair, smile, white dress, frill, very long dress, music live, Uyuni salt lake, walking on the water, Rembrandt lighting, colorful refraction, lens flare, bloom, film Reflection"
# negative_prompt = "lowres, bad anatomy, bad hands, text, error, missing fingers, extra digit, fewer digits,  multiple legs, malformation, close up, big breasts"

# seedをここで設定しておく。プロンプト変えて再実行のとき同じseedで再現性を確保するため。
seed = 123

In [ ]:
print("prompt          :", prompt)
print("negative_prompt :", negative_prompt)
print("seed            :", seed)
print("size            :", f"{width} x {height}")
print("steps           :", num_inference_steps)
print("guidance_scale  :", guidance_scale)

### 生成

`torch.Generator` で seed を固定して再現性を確保します。
MPS は `Generator(device='mps')` を直接サポートしていないため、generator は CPU で作って渡します。

In [ ]:
import time
from pathlib import Path

# 生成画像の外部保存先 (notebook 内 cell 表示と同時に PNG として書き出す)
# notebook は lecture/ サブディレクトリから実行される前提 → 親ディレクトリ outputs/ を使う
NB_OUT_DIR = (Path("outputs/00_sd15_intro") if IN_COLAB else Path("../outputs/00_sd15_intro"))
NB_OUT_DIR.mkdir(parents=True, exist_ok=True)

generator = torch.Generator(device="cpu" if device == "mps" else device).manual_seed(seed)

print("生成中 ...")
t0 = time.perf_counter()
result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt or None,
    width=width,
    height=height,
    num_inference_steps=num_inference_steps,
    guidance_scale=guidance_scale,
    generator=generator,
)
elapsed = time.perf_counter() - t0
print(f"done in {elapsed:.2f} s")

# pipe.__call__ には戻り値型注釈が無く、Pyright は tuple との union と推論する。
# ここでは return_dict=True (既定) なので StableDiffusionPipelineOutput が返り、.images が使える。
image = result.images[0]  # pyright: ignore[reportAttributeAccessIssue]
out_path = NB_OUT_DIR / "sec3_base_40steps.png"
image.save(out_path)
print(f"saved: {out_path}")
image

**生成時間と device 比較**

上のセルで実測されたのが、この環境での生成時間です（512×512 / 40 steps / fp32）。UNet を `num_inference_steps` 回呼ぶので、steps を倍にすると概ね時間も倍になります。

### CPU でも同じ画像を生成して時間を比較する

比較のために、pipeline を一時的に CPU に移して同じ prompt / seed で生成します。CPU では fp16 が遅い・一部 op 未対応なので、CUDA 環境 (fp16) の場合は **CPU 上で fp32 にキャスト**してから走らせます。CPU は GPU / MPS より大幅に遅く、Mac の M4 Max CPU で約 50 秒、非力な環境（Colab 無料 CPU 等）では十数分かかることもあります。

**3 環境の実測比較** (SD1.5 / 512×512 / 40 steps / 同一 prompt・seed の 1 枚生成):

| 環境 | device | dtype | 生成時間 |
|---|---|---|---:|
| Mac (M4 Max) | MPS (GPU) | fp32 | 9.95 s |
| Colab 無料 | T4 (GPU) | fp16 | 9.91 s |
| Windows (RTX 5080) | CUDA (GPU) | fp16 | 1.9 s (初回) / 1.5 s (定常) |
| Mac (M4 Max) | CPU | fp32 | 48.19 s |
| Colab 無料 | CPU (2 vCPU) | fp32 | 787.55 s (約 13 分) |
| Windows (Ryzen 7 9800X3D) | CPU | fp32 | 94.70 s |

観察:

- **無料 Colab の T4 ≈ Mac M4 Max の MPS** (ともに約 9.9 秒) — クラウドの無料 GPU がローカル高級機と互角。
- **GPU vs CPU の桁違い**: 同じ Colab で GPU 9.91 s ↔ CPU 787.55 s ＝ **約 79 倍**。デバイスを変えるだけでこの差。
- **同じ "CPU" でも環境差が大きい**: Mac 約 48 秒 ↔ Colab 無料 CPU 約 13 分 (割り当て 2 vCPU が非力)。「CPU なら何でも同じ」ではない。
- Windows (RTX 5080) は GPU 定常 ~1.5 秒が最速 (warmup 込みの初回は ~1.9 秒)。

steps を変えると MPS / CPU 双方の生成時間がほぼ線形にスケール (steps が UNet 呼び出し回数で律速)。CPU/MPS の比率自体は system 負荷や workload 構成で実行ごとに 3〜6 倍程度の範囲で変動します。

In [ ]:
# pipe を一時的に CPU に移して同じ条件で生成し、所要時間を比較する。
# - mps / cuda 環境では device != "cpu" なので比較が意味を持つ
# - cuda は load 時に fp16 を選んでいるので、CPU 上では fp32 にキャストして動かす
#   (CPU では fp16 が遅い・一部 op 未対応のため)
# - 既に device == "cpu" なら §3 と同じになるのでスキップ

if device == "cpu":
    print("[skip] 現在の device は既に CPU です。§3 の実測がそのまま CPU 時間です。")
    elapsed_cpu = elapsed
    image_cpu = None
else:
    print(f"[device] pipe を {device} ({dtype}) -> cpu (fp32) に移動 ...")
    pipe.to("cpu", torch.float32)

    cpu_generator = torch.Generator(device="cpu").manual_seed(seed)
    print("CPU で生成中 ... (数分かかることがあります)")
    t0 = time.perf_counter()
    result_cpu = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt or None,
        width=width,
        height=height,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=cpu_generator,
    )
    elapsed_cpu = time.perf_counter() - t0
    print(f"done in {elapsed_cpu:.2f} s  (CPU, fp32)")
    print(f"参考: 上のセルでの {device} 実測は {elapsed:.2f} s  →  CPU は約 {elapsed_cpu / elapsed:.1f} 倍遅い")

    # 後続セル (§7 の generate ラッパー) のために元の device / dtype に戻す
    print(f"[device] pipe を cpu -> {device} ({dtype}) に戻す ...")
    pipe.to(device, dtype)

    image_cpu = result_cpu.images[0]  # pyright: ignore[reportAttributeAccessIssue]
    out_path = NB_OUT_DIR / "sec3_base_40steps_cpu.png"
    image_cpu.save(out_path)
    print(f"saved: {out_path}")

image_cpu

<a id="peek"></a>

---
## 4. パイプラインを覗いてみる

`pipe.components` には、このパイプラインが内部で持っている主要コンポーネントが入っています。

In [ ]:
pipe.components

In [ ]:
import torch.nn as nn

# 生成パイプラインの中核 3 モジュール (intro で「およそ 1B」と紹介したもの)
core_modules = {"text_encoder", "unet", "vae"}
core_total = 0
all_total = 0

for name, obj in pipe.components.items():
    cls = type(obj).__name__ if obj is not None else "None"
    if isinstance(obj, nn.Module):
        n_params = sum(p.numel() for p in obj.parameters())
        all_total += n_params
        if name in core_modules:
            core_total += n_params
        tag = "← 生成中核" if name in core_modules else ""
        print(f"  {name:20s} : {cls:32s}  {n_params/1e6:>7.1f} M  {tag}")
    else:
        print(f"  {name:20s} : {cls:32s}        -")

print()
print(f"  生成中核 (text_encoder + UNet + VAE) : {core_total/1e6:>7.1f} M  (≈ {core_total/1e9:.2f} B)")
print(f"  全 nn.Module 合計                    : {all_total/1e6:>7.1f} M")

主な役割:

| コンポーネント | 役割 |
|---|---|
| `tokenizer` | prompt 文字列を token id 列に変換（CLIP tokenizer） |
| `text_encoder` | token 列を text embedding に変換（CLIPTextModel） |
| `unet` | latent から noise を予測する denoiser（UNet2DConditionModel） |
| `vae` | pixel ↔ latent の変換（AutoencoderKL、空間 1 辺で 8 倍 down/up sample） |
| `scheduler` | diffusion の step を進める（SD1.5 default は PNDMScheduler） |
| `safety_checker` | NSFW 画像検出（CLIP ベース） |
| `feature_extractor` | safety_checker 用の前処理 |

**パラメータ規模**: 上の実測値から、intro で「およそ 1B」と紹介した内訳は **text_encoder ≈ 123 M + UNet ≈ 860 M + VAE ≈ 84 M = 約 1.07 B** です。**UNet が 80% を占める**主役で、text encoder と VAE は脇役。`safety_checker` (≈ 304 M) は CLIP ベースの NSFW 判定器で、生成自体には参加しないので生成性能を語るときは数えません。`tokenizer` / `scheduler` / `feature_extractor` は nn.Module ではなく設定だけを持つ small Python class なので、param 数は無し (上の "−" 行)。


### 全体の流れ（latent diffusion の処理）

```
prompt (text)
  ↓ tokenizer + CLIPTextModel           # text を embedding に変換
text embeddings (1, 77, 768)
  ↓ scheduler が denoising の時刻列と更新式を決める
  ↓ 各 step で UNet が「今の latent に乗っているノイズ」を予測 → scheduler の更新式で latent を更新
clean latent (1, 4, 64, 64)
  ↓ VAE decoder                          # latent → pixel
画像 (3, 512, 512)
```

UNet は **同じ重みを T 回呼ぶ**（言語モデルが 1 回の forward で全 token を処理するのとは対照的）。


### モデルのバイト数を予測して disk cache と照合する

「パラメータ数 × dtype あたりのバイト数」から **総バイト数を予測** し、HF cache 上の重みファイル (`.safetensors`) の物理サイズと照合します。

dtype あたりのバイト数:

- **fp32**: 1 param = 4 bytes
- **fp16 / bf16**: 1 param = 2 bytes
- **int8** (量子化): 1 param = 1 byte

disk 上の dtype と in-memory の dtype (`torch_dtype` 引数) が **一致していれば** `ratio = disk / predicted ≈ 1.0`。

ratio が予想から外れる代表的なケース:

- `ratio ≈ 0.5`: cache が fp16 配布、`torch_dtype=torch.float32` で in-memory は fp32 (load 時にキャスト)
- `ratio 0.9〜0.95`: `parameters()` と state_dict / `.safetensors` 格納内容の小さな差 (一部 buffer や tied weight 等) — 通常無視可
- `(n/a)`: cache に該当 component の重みファイルが無い (configs だけ等)

なお、ここで扱うのは **重み container のサイズだけ**。tokenizer の vocab.json、scheduler config などは別途数 MB 含まれますが下では計算に入れていません。

In [ ]:
from pathlib import Path
from huggingface_hub import try_to_load_from_cache

# 各 component の cache 上の重みファイル候補 (HF 慣習: diffusers は diffusion_pytorch_model.*、transformers は model.*)
WEIGHT_FILENAMES = (
    "diffusion_pytorch_model.safetensors",  # diffusers (UNet, VAE)
    "model.safetensors",                     # transformers (text_encoder, safety_checker)
    "diffusion_pytorch_model.bin",
    "model.bin",
)


def find_cache_size(repo_id: str, component: str) -> int:
    """各 component の重みファイルを HF cache から探して bytes を返す (見つからなければ 0)。"""
    for fname in WEIGHT_FILENAMES:
        result = try_to_load_from_cache(repo_id=repo_id, filename=f"{component}/{fname}")
        if isinstance(result, str) and Path(result).exists():
            return Path(result).stat().st_size
    return 0


bytes_per_param = torch.empty(0, dtype=dtype).element_size()
print(f"current dtype : {dtype}  ({bytes_per_param} bytes/param)")
print(f"cache repo    : {MODEL_ID}\n")

print(f"  {'component':18s}  {'params':>8s}  {'predicted':>11s}  {'disk file':>11s}  {'ratio':>6s}")
print(f"  {'-'*18}  {'-'*8}  {'-'*11}  {'-'*11}  {'-'*6}")

pred_total = 0
disk_total = 0
for name, obj in pipe.components.items():
    if not isinstance(obj, nn.Module):
        continue
    n_params = sum(p.numel() for p in obj.parameters())
    predicted = n_params * bytes_per_param
    disk = find_cache_size(MODEL_ID, name)
    pred_total += predicted
    disk_total += disk
    ratio_str = f"{disk/predicted:.2f}x" if predicted > 0 and disk > 0 else "(n/a)"
    print(f"  {name:18s}  {n_params/1e6:>6.1f} M  {predicted/(1024**2):>8.1f} MB  {disk/(1024**2):>8.1f} MB  {ratio_str:>6s}")

print(f"  {'-'*18}  {'-'*8}  {'-'*11}  {'-'*11}  {'-'*6}")
total_ratio = f"{disk_total/pred_total:.2f}x" if pred_total > 0 else "(n/a)"
print(f"  {'total':18s}    {'':>6s}    {pred_total/(1024**3):>8.3f} GB  {disk_total/(1024**3):>8.3f} GB  {total_ratio:>6s}")

# --- snapshot dir 全体のサイズ (component 区別なし、tokenizer / configs / model_index.json 等も含む) ---
# 上の find_cache_size は「各 component の重みファイル」しか拾わないので、それ以外を可視化する。
print()
sample = try_to_load_from_cache(repo_id=MODEL_ID, filename="model_index.json")
if isinstance(sample, str):
    snap_dir = Path(sample).parent
    # rglob + .stat() は symlink を追跡するので、blobs/ 配下の実体サイズが集計される
    cache_total = sum(p.stat().st_size for p in snap_dir.rglob("*") if p.is_file())
    others = cache_total - disk_total
    print(f"  snapshot dir 全体 (symlink 追跡、component 区別なし):")
    print(f"    cache 全体   : {cache_total/(1024**3):>7.3f} GB")
    print(f"    重み合計     : {disk_total/(1024**3):>7.3f} GB   (上の表の disk file 合計)")
    print(f"    その他       : {others/(1024**2):>7.2f} MB   (tokenizer / configs / scheduler / model_index.json 等)")
else:
    print("  (snapshot dir を解決できませんでした)")

### 観察: cache の構成と §2「約 5 GB」表記の整合性

`predicted` と `disk file` の各行・合計が **1.00x で完全一致**すれば、「重み合計 ≈ params × dtype size = disk cache file size」が成立しているということです。§2 冒頭で「約 5 GB を download」と書いたのはこの実測ベースの値です。

**M4 Max 実測リファレンス** (2026-05-24、SD1.5 fp32 default variant):

| 項目 | params | predicted | disk |
|---|---:|---:|---:|
| text_encoder | 123.1 M | 469.4 MB | 469.5 MB |
| unet | 859.5 M | 3278.8 MB | 3278.9 MB |
| vae | 83.7 M | 319.1 MB | 319.1 MB |
| safety_checker | 304.0 M | 1159.6 MB | 1159.7 MB |
| **重み合計 (4 components)** | **1370.2 M** | **5.104 GB** | **5.105 GB** |
| その他 (tokenizer / configs / scheduler / model_index.json) | – | – | ~1.5 MB |
| **cache 全体** (snapshot dir 総和) | – | – | **5.106 GB** |

(MB / GB はすべて binary、1 MB = 1024^2 bytes)

「その他 1.5 MB」の中身はほぼ tokenizer の vocab.json + merges.txt (合計約 1.51 MB) で、それ以外の `model_index.json` / 各 component の `config.json` / scheduler / feature_extractor の設定は数百 B 〜 数 KB と無視できる規模。**つまり SD1.5 の cache size は 99.97% が重みで占められ、残り 0.03% が設定 + tokenizer**、というのが実態です。

### `variant` と `torch_dtype`: disk と memory の dtype は独立に指定する

`from_pretrained(model_id, variant=..., torch_dtype=...)` には dtype 関連のオプションが 2 つあり、**全く別の役割**を持ちます:

| オプション | 何を指定するか | いつ働くか |
|---|---|---|
| `variant=...` | **disk から download する重みファイルの dtype** (HF Hub 上の variant 選択) | 初回 download のとき (cache hit 後は何もしない) |
| `torch_dtype=...` | **メモリに load するときの dtype**。disk と違えば load 中に cast (型変換) が入る | `from_pretrained()` 呼び出しのたび |

SD1.5 の場合、HF Hub には 2 つの variant が置かれています:

- `diffusion_pytorch_model.safetensors` (suffix なし) → **fp32** (default variant)
- `diffusion_pytorch_model.fp16.safetensors` (`.fp16` 付き) → **fp16** variant

組み合わせると 4 通り。in-memory の重み合計サイズ目安は:

| disk variant | `torch_dtype` | cast | disk | memory |
|---|---|---|---:|---:|
| default (= fp32) | `torch.float32` | なし | ~5 GB | ~5 GB |
| default (= fp32) | `torch.float16` | down-cast (load 時) | ~5 GB | **~2.5 GB** |
| `variant="fp16"` | `torch.float32` | up-cast (load 時) | **~2.5 GB** | ~5 GB |
| `variant="fp16"` | `torch.float16` | なし | ~2.5 GB | ~2.5 GB |

(disk 側 ~2.5 GB は fp16 = fp32 の半分という dtype 算術からの目安。今回の cache は default variant のみで 5.1 GB のため、fp16 variant は未実測)

覚え方: **ディスクを節約したい → `variant="fp16"`**、**メモリを節約したい → `torch_dtype=torch.float16`**。両方節約したいなら両方 fp16。

### なぜ §1 で device ごとに `dtype` が違うのか

このノートブック §1 環境確認では device に応じて `torch_dtype` のデフォルトを変えています:

- **CUDA**: `torch.float16` — NVIDIA GPU は fp16 tensor core を持ち fp16 演算が fp32 より高速、VRAM も半分になる。Diffusers + CUDA の標準パターン
- **MPS (Apple Silicon)**: `torch.float32` — MPS + fp16 で SD1.5 の `safety_checker` (CLIP) が誤発火して黒画像を返す既知問題があり、回避のため fp32
- **CPU**: `torch.float32` — CPU には fp16 専用命令がほぼ無く、fp16 演算は emulation で逆に遅い

`variant` は指定していないので、どの環境でも **disk は default variant = fp32 で約 5 GB**。`torch_dtype` の選択だけが環境ごとに変わります (上の組み合わせ表で言うと 1 〜 2 行目を選んでいる)。

CUDA 環境で disk 容量も節約したい場合は明示的に `from_pretrained(..., variant="fp16", torch_dtype=torch.float16)` を渡せば 4 行目 (disk も memory も ~2.5 GB) になります。

<a id="inside"></a>

---
## 5. パイプラインの中身を見る

各コンポーネントを少し詳しく観察します。

### text encoder（CLIPTextModel）

prompt を embedding に変換する Transformer。SD1.5 は **OpenAI CLIP ViT-L/14** の text 側を使います。

In [ ]:
text_encoder = pipe.text_encoder
text_cfg = text_encoder.config

print("class                :", type(text_encoder).__name__)
print("hidden_size          :", text_cfg.hidden_size)
print("num_hidden_layers    :", text_cfg.num_hidden_layers)
print("num_attention_heads  :", text_cfg.num_attention_heads)
print("max_position_embeddings:", text_cfg.max_position_embeddings)
print("vocab_size           :", text_cfg.vocab_size)

total = sum(p.numel() for p in text_encoder.parameters())
print(f"parameters           : {total / 1e6:.1f} M")

**CLIP text encoder の特徴**

- 12 層の Transformer、隠れ次元 768、attention head 数 12
- 入力長は最大 77 token（足りない部分は padding、超える部分は truncate）
- 出力は `(batch, 77, 768)` の embedding。これを UNet の cross-attention で参照する

### UNet（UNet2DConditionModel）

latent から「乗っているノイズ」を予測する diffusion の主役。

In [ ]:
unet = pipe.unet
unet_cfg = unet.config

print("class                :", type(unet).__name__)
print("sample_size          :", unet_cfg.sample_size, " (latent edge length)")
print("in_channels          :", unet_cfg.in_channels,  " (latent channel)")
print("out_channels         :", unet_cfg.out_channels, " (predicted noise channel)")
print("cross_attention_dim  :", unet_cfg.cross_attention_dim, " (= text embedding dim)")
print("down_block_types     :", unet_cfg.down_block_types)
print("up_block_types       :", unet_cfg.up_block_types)
print("block_out_channels   :", unet_cfg.block_out_channels)

total = sum(p.numel() for p in unet.parameters())
print(f"parameters           : {total / 1e6:.1f} M")

**UNet の特徴**

| 項目 | 値 | 意味 |
|---|---|---|
| `sample_size = 64` | latent 解像度 | 画像 512px は VAE で 1 辺 1/8 に縮められて 64px latent になる (面積で 1/64) |
| `in_channels = 4` | latent の channel 数 | VAE の出力次元 |
| `cross_attention_dim = 768` | text embedding の次元 | CLIP の `hidden_size` と一致 |
| `down_block_types` | encoder 側のブロック構成 | CrossAttn 付き ResNet が含まれる |
| `up_block_types` | decoder 側のブロック構成 | U 字の対称構造 |

UNet は **prompt embedding を cross-attention で参照しながら** noise を予測します。
これによって「prompt に従った」生成が可能になります。

### VAE（AutoencoderKL）

pixel ↔ latent の変換器。

In [ ]:
vae = pipe.vae
vae_cfg = vae.config

print("class              :", type(vae).__name__)
print("in_channels        :", vae_cfg.in_channels,  " (RGB)")
print("out_channels       :", vae_cfg.out_channels, " (RGB)")
print("latent_channels    :", vae_cfg.latent_channels)
print("block_out_channels :", vae_cfg.block_out_channels)
print("scaling_factor     :", vae_cfg.scaling_factor)

total = sum(p.numel() for p in vae.parameters())
print(f"parameters         : {total / 1e6:.1f} M")

**VAE の特徴**

- encoder: `(B, 3, 512, 512) → (B, 4, 64, 64)` — 空間 1 辺を 1/8 に縮小 (面積で 1/64)、channel は 3 → 4
- decoder: `(B, 4, 64, 64) → (B, 3, 512, 512)` — encoder の逆向き。空間 1 辺を 8 倍に拡大、channel は 4 → 3 に復元
- 値の総数は pixel 786,432 ⇔ latent 16,384 で **約 48 倍の圧縮**。各 latent 位置は元画像上のおおよそ 8×8 px 間隔のグリッドに対応し、4 次元ベクトルで表される (ただし VAE は畳み込みネットワークなので、厳密に独立した 8×8 patch に分割しているわけではなく、受容野は重なる)
- `scaling_factor` は VAE が出す生 latent を**単位分散** (std≈1) に正規化して diffusion が想定するスケールに合わせる定数（学習潜在の component-wise 標準偏差。`z*scaling_factor` で UNet へ、decode 前に `z/scaling_factor` で戻す。SD1.5 では 0.18215）

### scheduler

diffusion の step を進める係数を保持するクラス。SD1.5 の default は `PNDMScheduler`。

In [ ]:
scheduler = pipe.scheduler

print("class                :", type(scheduler).__name__)
print("num_train_timesteps  :", scheduler.config.num_train_timesteps)
print("beta_schedule        :", scheduler.config.beta_schedule)
print("prediction_type      :", scheduler.config.prediction_type)

**scheduler の役割**

- 学習時は `num_train_timesteps = 1000` の刻みでノイズを乗せていく
- 推論時はそこから `num_inference_steps` 個だけ「飛び石」で step を選んで denoise する
- 別の scheduler（DPMSolver, Euler 等）に差し替えるだけで生成速度・品質のトレードオフが変えられる

<a id="tokenizer"></a>

---
## 6. tokenizer の動作を確認

prompt がどのように token 列になるか確認します。CLIP tokenizer は **BPE (byte-pair encoding)** を使います。

In [ ]:
tokenizer = pipe.tokenizer

ids = tokenizer.encode(prompt)
print(f"prompt    : {prompt}")
print(f"token 数  : {len(ids)}")
print()
for i, tid in enumerate(ids):
    piece = tokenizer.decode([tid])
    print(f"  [{i:2d}] id={tid:6d}  '{piece}'")

**気づき**

- 先頭に `<|startoftext|>`、末尾に `<|endoftext|>` の special token が付く（CLIP の慣習）
- `witch` `forest` `magic` `staff` のような **よく出る単語は 1 token** に収まる
- `bioluminescent` のような **長い・珍しい単語は複数 token に分解**される (上の出力では `bi` / `olu` / `min` / `esc` / `ent` の 5 token に分解)
- 内部の BPE 辞書では各 token に「単語末尾」マーカー `</w>` が付いているが、`tokenizer.decode()` が表示時に取り除くため、上の出力には現れない
- CLIP tokenizer の最大長は `max_position_embeddings = 77` で、超過分は truncate (今回 33 token なのでまだ余裕あり)


<a id="sweep"></a>

---
## 7. seed と steps を変えてみる

同じ prompt でもパラメータが変わると出力が大きく変わります。

### seed を変える

seed が違うと初期 noise が違うので、構図が変わります。

In [ ]:
def generate(
    *,
    prompt: str,
    negative_prompt: str | None = None,
    seed: int,
    num_inference_steps: int,
    guidance_scale: float = 7.5,
):
    """小さなラッパー。seed と steps を変えて試すために用意する。"""
    g = torch.Generator(device="cpu" if device == "mps" else device).manual_seed(seed)
    out = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt or None,
        width=512,
        height=512,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=g,
    )
    return out.images[0]  # pyright: ignore[reportAttributeAccessIssue]


img = generate(
    prompt=prompt,
    negative_prompt=negative_prompt,
    seed=seed + 100,
    num_inference_steps=num_inference_steps,
)
out_path = NB_OUT_DIR / "sec7_seed_plus100.png"
img.save(out_path)
print(f"saved: {out_path}")
img

### steps を変える

`num_inference_steps` を小さくすると denoise が荒くなり、大きくすると細部が安定します。
ただし大きくしすぎても飽和するため、SD1.5 では 20〜50 が実用範囲です。

In [ ]:
img = generate(
    prompt=prompt,
    negative_prompt=negative_prompt,
    seed=seed,
    num_inference_steps=num_inference_steps // 2,
)
out_path = NB_OUT_DIR / "sec7_steps_half.png"
img.save(out_path)
print(f"saved: {out_path}")
img

In [ ]:
img = generate(
    prompt=prompt,
    negative_prompt=negative_prompt,
    seed=seed,
    num_inference_steps=num_inference_steps // 4,
)
out_path = NB_OUT_DIR / "sec7_steps_quarter.png"
img.save(out_path)
print(f"saved: {out_path}")
img

**比較**

§3 の base `num_inference_steps` (現在 40) を基準に、`// 2` と `// 4` で減らしたケースと並べて比較します:

- **base = 40 steps** (§3 で生成済): 細部が安定。anime FT 系の顔精度もここで揃う
- **`// 2 = 20` steps**: 一段細部が荒くなる。SD1.5 base ではこのあたりが実用下限
- **`// 4 = 10` steps**: ノイズが残ったり、細部 (顔・手) が大きく崩れることが多い

一般的に SD1.5 では **20〜50 が実用範囲**。anime FT 系は顔の精度に厳しいので 40〜50 にすると安定。生成時間は steps にほぼ線形に比例 (= UNet を steps 回呼ぶため)。

`guidance_scale` を変えると prompt への忠実度が変わります（高いほど prompt 通りだが多様性が減る）。興味があれば `generate(..., guidance_scale=12.0)` 等を試してみてください。

<a id="alts"></a>

---
## 8. 他のモデルを試す (SD1.5 系の FT モデル)

SD1.5 のアーキテクチャを共有しつつ、画風を変えて **fine-tune (FT)** した派生モデルが Hugging Face に大量に公開されています。`StableDiffusionPipeline` の `model_id` を差し替えるだけで切り替わるので、ここでは元の SD1.5 と並べて生成してみます。

別の変数 `pipe_alt` で読み込み、§3 の `pipe` は壊さないようにします。

下の cell には **3 年前 (2023 年) の自分の試行 notebook で並べていた model_id** をそのまま残し、各行の末尾に **2026 年現在の HF Hub での生存状況** をコメントしてあります。好きなものを 1 つ uncomment してください。

> **注意**: FT モデルはモデルごとにライセンス、学習データの由来、NSFW 傾向が異なります。例えば [`nitrosocke/Arcane-Diffusion`](https://huggingface.co/nitrosocke/Arcane-Diffusion) は TV 作品 "Arcane" のスタイルを学習したモデル、[`dreamlike-art/dreamlike-photoreal-2.0`](https://huggingface.co/dreamlike-art/dreamlike-photoreal-2.0) は model card に NSFW 寄りの注意書きがあります。デモで使う場合は、配布条件と安全性を model card で確認したものだけを選んでください。


In [ ]:
# 候補から 1 つだけ uncomment する。
# 注: 各 FT モデルは ~4 GB の重みを download するので、初回は時間がかかります。

# --- 元の SD1.5 (上の pipe と同じ重み、確認用) ---
# alt_model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

# --- 2023 年の inbox notebook で試した model_id (2026-05 時点の生存確認結果コメント付き) ---
# alt_model_id = "runwayml/stable-diffusion-v1-5"   # ▲ RunwayML repo は削除。HF 上は stable-diffusion-v1-5/... にリダイレクト
# alt_model_id = "prompthero/openjourney"             # ✓ HF 健在 / Midjourney 風 FT
# alt_model_id = "andite/anything-v4.0"             # ✗ 2026-05 時点で直接参照不可 (HTTP 401)。401 だけでは削除/非公開/gated/認証エラーは区別できないので原因は断定しない。下の xyn-ai/ が community 再アップ (model card に "Duplicate from andite/anything-v4.0" 表示)

# --- 現在も healthy な SD1.5 系 FT モデル ---
alt_model_id = "xyn-ai/anything-v4.0"             # ✓ anime 風 (andite の再アップ)
# alt_model_id = "Linaqruf/anything-v3.0"           # ✓ anime 風 (v3 系)
# alt_model_id = "nitrosocke/Arcane-Diffusion"      # ✓ Netflix Arcane のアートスタイル
# alt_model_id = "dreamlike-art/dreamlike-photoreal-2.0"  # ✓ photoreal 系

print(f"loading: {alt_model_id}  (初回は ~4 GB download なので時間がかかります)")
t_load_alt = time.perf_counter()
pipe_alt = StableDiffusionPipeline.from_pretrained(alt_model_id, torch_dtype=dtype)
pipe_alt = pipe_alt.to(device)
# FT モデルは safety_checker 未調整のことが多く、健全な prompt でも誤発火 (黒画像) しやすいので外す
pipe_alt.safety_checker = None
if device == "mps":
    pipe_alt.enable_attention_slicing()
load_elapsed_alt = time.perf_counter() - t_load_alt
print(f"load done in {load_elapsed_alt:.2f} s")
print("pipeline クラス:", type(pipe_alt).__name__)


In [ ]:
# §8 の alt model 生成用に prompt / negative_prompt / seed を再設定。
# §3 と同じ設定にしておくと「同じ prompt で base vs FT」の比較になる。
# 片方だけ変えて遊んでも OK (例: §3 を horse、§8 を witch にして見比べる等)。

# 古典的な SD1.5 デモプロンプト (参考):
# prompt = "a photo of an astronaut riding a horse"
# negative_prompt = ""

# 魔法使い (自然言語版)
prompt = (
    "a young witch with long flowing hair holding a glowing magic staff, "
    "in an enchanted forest with bioluminescent mushrooms, "
    "fantasy art, masterpiece, detailed"
)
negative_prompt = (
    "blurry, low quality, distorted, bad anatomy, extra limbs, "
    "watermark, signature"
)

# 魔法使い (tag 列挙版)
# prompt = (
#     "witch with magic staff, long hair, enchanted forest, glowing mushrooms, "
#     "bloom, high quality, very high resolution, colorful refraction, lens flare"
# )
# negative_prompt = (
#     "bad anatomy, bad hands, text, error, missing fingers, extra digit, "
#     "extra tail, fewer digits, multiple legs, malformation, close up"
# )

# 以前あったもの
# prompt = "high quality, very high resolution, large filesize, high detailed, distant wide shot, straight hair, dark hair, very long hair, smile, white dress, frill, very long dress, music live, Uyuni salt lake, walking on the water, Rembrandt lighting, colorful refraction, lens flare, bloom, film Reflection"
# negative_prompt = "lowres, bad anatomy, bad hands, text, error, missing fingers, extra digit, fewer digits,  multiple legs, malformation, close up, big breasts"

num_inference_steps = 40
seed = 123

In [ ]:
# §3 で作った generator を再 seed して使い回す (新規 Generator は不要)。
# generator は state machine で、一度生成に使うと内部状態が進むので、
# 同じ seed の初期 noise から再スタートしたいときは manual_seed(seed) で巻き戻す。
generator.manual_seed(seed)

print(f"生成中 with {alt_model_id} ...")
t0 = time.perf_counter()
result_alt = pipe_alt(
    prompt=prompt,
    negative_prompt=negative_prompt or None,
    width=width,
    height=height,
    num_inference_steps=num_inference_steps,
    guidance_scale=guidance_scale,
    generator=generator,
)
elapsed_alt = time.perf_counter() - t0
print(f"done in {elapsed_alt:.2f} s")

image_alt = result_alt.images[0]  # pyright: ignore[reportAttributeAccessIssue]
# ファイル名は alt_model_id の末尾 (基底名) から組み立て (例: 'xyn-ai/anything-v4.0' -> 'anything-v4.0')
alt_basename = alt_model_id.split("/")[-1]
out_path = NB_OUT_DIR / f"sec8_alt_{alt_basename}.png"
image_alt.save(out_path)
print(f"saved: {out_path}")
image_alt


**比較**

同じ prompt / seed / steps / guidance でも、ベースの FT モデルが違うと画風が変わります:

- 元の SD1.5: prompt をそのまま描く、平均的な写実寄り
- [`prompthero/openjourney`](https://huggingface.co/prompthero/openjourney): Midjourney 風 (色が濃い・コンセプトアート寄り)
- [`xyn-ai/anything-v4.0`](https://huggingface.co/xyn-ai/anything-v4.0) / [`Linaqruf/anything-v3.0`](https://huggingface.co/Linaqruf/anything-v3.0): anime 風
- [`nitrosocke/Arcane-Diffusion`](https://huggingface.co/nitrosocke/Arcane-Diffusion): Netflix Arcane のアートスタイル
- [`dreamlike-art/dreamlike-photoreal-2.0`](https://huggingface.co/dreamlike-art/dreamlike-photoreal-2.0): 写真風の細部

これは SD1.5 の **アーキテクチャ (UNet + CLIP + VAE) を共有**しつつ、それぞれ違う画像セットで fine-tune した重みを使っているからです。`from_pretrained()` で `model_id` を差し替えるだけで画風がガラッと変わる、というのが diffusion model の楽しいところ。

§3 で生成した元の SD1.5 の画像 (`image`) と、上の `image_alt` を並べて比べると、**同じ prompt でも全く違う絵**になっているはずです。


<a id="summary"></a>

---
## まとめ

| ステップ | 処理 | 主役 |
|---|---|---|
| tokenize | prompt → token id 列 | `tokenizer` (CLIP BPE) |
| text encode | token 列 → embedding (1, 77, 768) | `text_encoder` (CLIPTextModel) |
| denoise | T 回繰り返し noise を引く | `unet` + `scheduler` |
| decode | latent (1, 4, 64, 64) → 画像 (3, 512, 512) | `vae` decoder |

**SD1.5 の特徴を一行で**: CLIP text embedding を cross-attention で参照する **UNet** が、VAE が定義する **64x64 の latent 空間** で diffusion を行うモデル。

次のノートブックでは、より詳しく **テキストから画像が出来上がるまでの中間状態** や、**異なるモデル (SDXL / FLUX / SD3.5 / Qwen-Image)** との比較を見ていきます。